# **Mixture of Experts (MoE): Практика**

### Практика к уроку «Mixture of Experts (MoE)» модуля «Нелинейные интерпретируемые модели»

В уроке «Mixture of Experts (MoE)» мы сказали о MoE несколько вещей, которые стоит проверить на данных, а не принимать на веру:

1. с правдоподобием смеси (формула 1.3 статьи Jacobs, Jordan, Nowlan и Hinton, 1991) эксперты специализируются на своих областях данных, а с ошибкой общего прогноза (формула 1.1) кооперируют;
2. gating-модель учится предсказывать ответственность эксперта $h_k(x,y)$, но видит только $x$, поэтому ее вес $g_k(x)$ мягче там, где области сходятся;
3. маршрутизация говорит, кто отвечает за объект, а почему прогноз такой — говорит сам эксперт;
4. частота выбора эксперта не равна его важности, и важность измеряют удалением эксперта;
5. в разреженной top-$k$ маршрутизации (Shazeer et al., 2017) нулевой вес значит «не прошел отбор», а ровную загрузку экспертов задает функция потерь балансировки, а не данные.

Все проверим на синтетике: два признака, три области с разными законами, линейные эксперты. Данные создаются в самой тетради, сиды фиксированы, и вся тетрадь считается на процессоре за пару минут.

### Какие библиотеки мы используем и почему не другие

| Библиотека | Что в ней есть | Что берем |
|---|---|---|
| `torch` | слои `nn.Linear`, `softmax`, `topk`, автоматическое дифференцирование | всю MoE: экспертов, gating-модель, обе функции потерь и noisy top-$k$ — по формулам урока «Mixture of Experts (MoE)», по несколько строк на каждую часть |
| `matplotlib` | графики | карту маршрутизации и график «частота против важности» |
| `scikit-learn` | `GaussianMixture` — смесь распределений $p(x)$, без экспертов и без gating-модели | не берем: MoE моделирует $p(y \mid x)$, а готовой MoE в scikit-learn нет |
| `transformers` | MoE-слои внутри больших языковых моделей: Mixtral, Switch Transformers | не берем: маршрутизацию там видно только на целой языковой модели, а нам нужны данные, где области и законы в них известны заранее |

`torch` и `matplotlib` уже стоят в Colab, ставить ничего не нужно.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

T0 = time.time()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SIGMA = 0.5            # σ в правдоподобии эксперта N(y; f_k(x), σ²), формула (1.3)
STEPS, LR = 3000, 0.05 # полный батч, Adam
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#17becf', '#bcbd22', '#7f7f7f']
print('устройство:', DEVICE, '| torch', torch.__version__)

## 1. Данные: три области с разными законами

Признаки $x = (x_1, x_2)$ равномерно распределены на квадрате $[-1, 1]^2$. Область задает $x_1$, и в каждой области свой закон:

- область 0, $x_1 < 0.4$ (около 70% объектов): $y = 3(x_1 + 0.3)^2 + x_2$ — парабола по $x_1$ с вершиной в $x_1 = -0.3$;
- область 1, $0.4 \le x_1 < 0.8$ (около 20%): $y = -2 + 2x_2$;
- область 2, $x_1 \ge 0.8$ (около 10%): $y = 4 - 2x_2$.

К ответу добавлен шум $\mathcal N(0, 0.1^2)$. В обучающей выборке 3000 объектов, в тестовой 2000, у каждой свой сид.

Законы областей 1 и 2 линейный эксперт опишет точно, а параболу области 0 одной прямой не описать. Поэтому заранее неизвестно, сколько экспертов достанется области 0: это мы и посмотрим.

In [ ]:
B1, B2 = 0.4, 0.8      # границы областей по x1

def make_data(n, seed):
    """x ~ U[-1,1]^2; три области по x1, в каждой свой закон y(x), шум N(0, 0.1^2)."""
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(n, 2, generator=g) * 2 - 1
    region = (x[:, 0] >= B1).long() + (x[:, 0] >= B2).long()
    laws = torch.stack([3 * (x[:, 0] + 0.3) ** 2 + x[:, 1],   # область 0: парабола по x1
                        -2 + 2 * x[:, 1],                      # область 1
                        4 - 2 * x[:, 1]], 1)                   # область 2
    y = laws.gather(1, region[:, None])[:, 0] + 0.1 * torch.randn(n, generator=g)
    return x.to(DEVICE), y.to(DEVICE), region.to(DEVICE)

x_tr, y_tr, r_tr = make_data(3000, seed=1)
x_te, y_te, r_te = make_data(2000, seed=2)
print('доли областей 0, 1, 2 в обучающей выборке:', [round(v, 3) for v in (torch.bincount(r_tr) / len(r_tr)).tolist()])

fig, ax = plt.subplots(figsize=(5.5, 4.4))
sc = ax.scatter(x_tr[:, 0].cpu(), x_tr[:, 1].cpu(), c=y_tr.cpu(), s=6, cmap='coolwarm')
for b in (B1, B2):
    ax.axvline(b, color='k', ls='--', lw=1)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.set_title('Обучающая выборка: цвет — y')
fig.colorbar(sc, label='$y$')
plt.show()

## 2. Классическая MoE: две функции потерь

Модель — как в постановке урока: $K = 4$ линейных эксперта $f_k(x) = w_k^\top x + b_k$ и линейная gating-модель $g(x) = \mathrm{softmax}(W_g x + c)$, прогноз смеси $\hat y(x) = \sum_k g_k(x) f_k(x)$. На старте веса gating-модели нулевые, и $g_k(x) = 1/4$ у всех экспертов.

Сравним две функции ошибки из раздела 1 статьи Jacobs, Jordan, Nowlan и Hinton (1991):

$$E_{1.1} = \Big(y - \sum_k g_k(x)\, f_k(x)\Big)^2, \qquad E_{1.3} = -\log \sum_k g_k(x)\, \exp\Big(-\frac{(y - f_k(x))^2}{2\sigma^2}\Big).$$

В статье $\sigma = 1$; у нас $\sigma = 0.5$. Это гиперпараметр, и в задании 1 его меняют.

Кроме MSE смеси считаем на тесте еще две величины. **MSE одного эксперта** — ошибка, если для каждого объекта взять прогноз одного эксперта с наибольшим весом $g_k(x)$: у специализированных экспертов она не больше ошибки смеси. **Ожидаемая ошибка (1.2)** — среднее $\sum_k g_k(x)\,(y - f_k(x))^2$: ошибка, если эксперта для объекта выбирать случайно с вероятностями $g_k(x)$.

В таблице под каждой моделью — веса экспертов ($b$, $w_1$, $w_2$), частота (доля объектов теста, где у эксперта наибольший вес $g_k(x)$) и доли объектов каждой области, которые gating-модель отдает этому эксперту.

In [ ]:
class MoE(nn.Module):
    """K линейных экспертов f_k(x) = w_k x + b_k и линейная gating-модель g(x) = softmax(W_g x + c)."""
    def __init__(self, K):
        super().__init__()
        self.experts = nn.Linear(2, K)       # строка k — эксперт k
        self.gate = nn.Linear(2, K)          # логиты gating-модели
        nn.init.zeros_(self.gate.weight)     # на старте g_k(x) = 1/K: все эксперты равны
        nn.init.zeros_(self.gate.bias)

    def forward(self, x):
        return self.experts(x), F.log_softmax(self.gate(x), -1)   # прогнозы f_k(x) и log g_k(x)


def loss_mixture(f, log_g, y, sigma):
    """(1.3): -log sum_k g_k(x) N(y; f_k(x), sigma^2), без постоянного слагаемого."""
    return -torch.logsumexp(log_g - (y[:, None] - f) ** 2 / (2 * sigma ** 2), -1).mean()


def loss_blend(f, log_g, y, sigma):
    """(1.1): квадрат ошибки общего прогноза sum_k g_k(x) f_k(x)."""
    return ((y - (log_g.exp() * f).sum(-1)) ** 2).mean()


def train(loss, K=4, seed=0, sigma=SIGMA, steps=STEPS, lr=LR):
    torch.manual_seed(seed)
    model = MoE(K).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        f, log_g = model(x_tr)
        opt.zero_grad()
        loss(f, log_g, y_tr, sigma).backward()
        opt.step()
    return model


@torch.no_grad()
def scores(model):
    """MSE на тесте: смеси, одного эксперта с наибольшим g_k(x) и ожидаемая ошибка одного эксперта (1.2)."""
    f, log_g = model(x_te)
    g = log_g.exp()
    top = g.argmax(-1)
    return dict(mix=((g * f).sum(-1) - y_te).pow(2).mean().item(),
                top=(f.gather(1, top[:, None])[:, 0] - y_te).pow(2).mean().item(),
                e12=(g * (y_te[:, None] - f) ** 2).sum(-1).mean().item())


@torch.no_grad()
def describe(model, name):
    """Качество, веса экспертов и доли объектов каждой области, которые gating-модель отдает эксперту."""
    f, log_g = model(x_te)
    top = log_g.argmax(-1)
    K = f.shape[1]
    s = scores(model)
    print(f"{name}: MSE смеси {s['mix']:.3f}, MSE одного эксперта с наибольшим весом {s['top']:.3f}, ожидаемая ошибка (1.2) {s['e12']:.3f}")
    print('  эксперт   b      w1     w2   частота  доли областей 0, 1, 2')
    for k in range(K):
        b, w = model.experts.bias[k].item(), model.experts.weight[k].tolist()
        shares = [((top == k) & (r_te == r)).sum().item() / (r_te == r).sum().item() for r in range(3)]
        print(f'  {k}   {b:6.2f} {w[0]:6.2f} {w[1]:6.2f}   {(top == k).float().mean().item():.3f}   '
              + '  '.join(f'{v:.2f}' for v in shares))


mix = train(loss_mixture)
blend = train(loss_blend)
describe(mix, 'функция потерь (1.3)')
print()
describe(blend, 'функция потерь (1.1)')

Если эксперт выучил закон своей области, его строка повторяет этот закон: у области 1 это $b = -2$, $w_1 = 0$, $w_2 = 2$, у области 2 — $b = 4$, $w_1 = 0$, $w_2 = -2$. Тогда объяснение двухуровневое, как в уроке: gating-модель говорит, кто отвечает за объект, а веса эксперта — почему прогноз такой. Найдите такие строки у обеих моделей и посмотрите, как поделена парабола области 0.

Один сид — один прогон. Повторим обе функции потерь на пяти сидах: сид меняет начальные веса экспертов, данные те же.

In [ ]:
print('сид   (1.3): смесь  один   (1.2)     (1.1): смесь  один   (1.2)')
for seed in range(5):
    a, b = scores(train(loss_mixture, seed=seed)), scores(train(loss_blend, seed=seed))
    print(f"  {seed}    {a['mix']:.3f}  {a['top']:.3f}  {a['e12']:.3f}     {b['mix']:.3f}  {b['top']:.3f}  {b['e12']:.3f}")

При какой функции потерь прогноз одного эксперта не хуже прогноза смеси на всех пяти сидах? При каком смесь точнее, а эксперты по отдельности ошибаются сильнее? Это и есть специализация и кооперация из раздела 1 статьи.

## 3. Ответственность $h_k(x, y)$ против весов gating $g_k(x)$

$$h_k(x, y) = \frac{g_k(x)\, \mathcal N\big(y;\, f_k(x),\, \sigma^2\big)}{\sum_j g_j(x)\, \mathcal N\big(y;\, f_j(x),\, \sigma^2\big)}$$

— апостериорная вероятность эксперта $k$, когда ответ $y$ уже известен. Gating-модель видит только $x$. Сравним обе величины на тесте у модели с функцией потерь (1.3): в среднем по всем объектам и у границы областей 1 и 2, где $|x_1 - 0.8| < 0.05$.

In [ ]:
@torch.no_grad()
def responsibility(model, x, y, sigma=SIGMA):
    """Ответственность h_k(x, y): апостериорная вероятность эксперта k, когда ответ y известен."""
    f, log_g = model(x)
    return F.softmax(log_g - (y[:, None] - f) ** 2 / (2 * sigma ** 2), -1)


with torch.no_grad():
    f, log_g = mix(x_te)
g, h = log_g.exp(), responsibility(mix, x_te, y_te)
near = (x_te[:, 0] - B2).abs() < 0.05
print(f'среднее max_k g_k / max_k h_k по тесту: {g.max(-1).values.mean().item():.3f} / {h.max(-1).values.mean().item():.3f}')
print(f'то же у границы x1 = 0.8 (объектов: {near.sum().item()}): {g[near].max(-1).values.mean().item():.3f} / {h[near].max(-1).values.mean().item():.3f}')
print(f'доля объектов, где argmax g и argmax h разные: {(g.argmax(-1) != h.argmax(-1)).float().mean().item():.4f}')
by_h = h.argmax(-1)
print(f'MSE одного эксперта с наибольшей ответственностью h_k: {(f.gather(1, by_h[:, None])[:, 0] - y_te).pow(2).mean().item():.3f}')

i = int(near.nonzero()[0])
print(f'\nОдин объект у границы: x = ({x_te[i, 0].item():.3f}, {x_te[i, 1].item():.3f}), y = {y_te[i].item():.2f}')
print('  g(x)   =', [round(v, 3) for v in g[i].tolist()])
print('  h(x,y) =', [round(v, 3) for v in h[i].tolist()])
print('  f(x)   =', [round(v, 2) for v in f[i].tolist()])

В уроке под катом «Строго: градиенты функции потерь MoE» выведено $\partial \ell / \partial s_k = g_k - h_k$, где $s_k$ — логиты gating-модели. Проверим это автоматическим дифференцированием на пяти объектах: разница должна быть на уровне ошибки округления float32.

In [ ]:
x5, y5 = x_tr[:5], y_tr[:5]
s = mix.gate(x5).detach().requires_grad_(True)       # логиты s_k(x) gating-модели
f5 = mix.experts(x5).detach()
loss_mixture(f5, F.log_softmax(s, -1), y5, SIGMA).mul(len(x5)).backward()   # сумма функций потерь пяти объектов
gap = (s.grad - (s.softmax(-1) - responsibility(mix, x5, y5))).abs().max().item()
print(f'max |dl/ds_k - (g_k - h_k)| = {gap:.1e}')

## 4. Карта маршрутизации

Покрасим квадрат $[-1, 1]^2$ по эксперту с наибольшим весом $g_k(x)$; чем бледнее цвет, тем меньше этот вес. Пунктир — настоящие границы областей $x_1 = 0.4$ и $x_1 = 0.8$. Под картой — точки на прямой $x_2 = 0$, где gating-модель с функцией потерь (1.3) переключает эксперта.

In [ ]:
xs = torch.linspace(-1, 1, 401, device=DEVICE)
X1, X2 = torch.meshgrid(xs, xs, indexing='xy')
grid = torch.stack([X1.flatten(), X2.flatten()], 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (name, model) in zip(axes, [('Маршрутизация, функция потерь (1.3)', mix), ('Маршрутизация, функция потерь (1.1)', blend)]):
    with torch.no_grad():
        g = model.gate(grid).softmax(-1)
    K = g.shape[1]
    rgb = torch.tensor([[int(c[j:j + 2], 16) / 255 for j in (1, 3, 5)] for c in COLORS[:K]])
    img = rgb[g.argmax(-1).cpu()].reshape(401, 401, 3)
    alpha = g.max(-1).values.cpu().reshape(401, 401, 1)
    ax.imshow(torch.cat([img, alpha], -1), origin='lower', extent=(-1, 1, -1, 1))
    for b in (B1, B2):
        ax.axvline(b, color='k', ls='--', lw=1)
    ax.set_title(name); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(handles=[Patch(color=COLORS[k], label=f'эксперт {k}') for k in range(K)], loc='lower left', fontsize=8)
plt.tight_layout(); plt.show()

line = torch.stack([torch.linspace(-1, 1, 2001, device=DEVICE), torch.zeros(2001, device=DEVICE)], 1)
with torch.no_grad():
    a = mix.gate(line).argmax(-1)
print('Где gating-модель переключает эксперта на прямой x2 = 0:')
for j in (a[1:] != a[:-1]).nonzero()[:, 0].tolist():
    print(f'  x1 = {(line[j, 0] + line[j + 1, 0]).item() / 2:.3f}: эксперт {a[j].item()} -> эксперт {a[j + 1].item()}')

Сравните точки переключения с границами областей. Внутри области 0 настоящей границы нет, а переключение есть: где оно стоит относительно вершины параболы $x_1 = -0.3$? Специализация экспертов не обязана совпадать с областями, которые задал человек.

## 5. Удаление эксперта: частота выбора против важности

Частота выбора эксперта — доля объектов теста, где у него наибольший вес $g_k(x)$. Важность измеряем вмешательством, как в уроке: убираем эксперта (его логит становится $-\infty$, softmax считается по остальным) и смотрим, на сколько выросла MSE смеси на тесте.

In [ ]:
@torch.no_grad()
def mse_without(model, k, x, y):
    """MSE смеси на (x, y), если эксперта k убрать: его логит -inf, softmax по остальным."""
    f, _ = model(x)
    logits = model.gate(x)
    logits[:, k] = float('-inf')
    return ((logits.softmax(-1) * f).sum(-1) - y).pow(2).mean().item()


@torch.no_grad()
def ablation_table(model, name):
    f, log_g = model(x_te)
    top = log_g.argmax(-1)
    base = scores(model)['mix']
    rows = []
    for k in range(f.shape[1]):
        freq = (top == k).float().mean().item()
        rows.append((freq, k, mse_without(model, k, x_te, y_te) - base))
    print(f'{name}: MSE смеси со всеми экспертами {base:.3f}')
    print('  эксперт  частота  рост MSE без него')
    for freq, k, d in sorted(rows, reverse=True):
        print(f'  {k}       {freq:.3f}      {d:9.2f}')
    return rows


rows13 = ablation_table(mix, 'функция потерь (1.3)')
print()
rows11 = ablation_table(blend, 'функция потерь (1.1)')

fig, ax = plt.subplots(figsize=(5.5, 4))
for freq, k, d in rows13:
    ax.scatter(freq, d, s=80, color=COLORS[k])
    ax.annotate(f'эксперт {k}', (freq, d), textcoords='offset points', xytext=(6, 4))
ax.margins(0.2)
ax.set_xlabel('частота выбора (доля объектов теста)'); ax.set_ylabel('рост MSE без эксперта'); ax.set_title('Функция потерь (1.3): частота против важности')
plt.show()

Упорядочены ли эксперты по важности так же, как по частоте? И что значат огромные числа у модели с функцией потерь (1.1)?

## 6. Sparse top-$k$ gating (Shazeer et al., 2017)

В трансформер-MoE gating-модель (роутер) оставляет объекту только $k$ экспертов:

$$G(x) = \mathrm{softmax}\big(\mathrm{KeepTopK}(H(x), k)\big), \qquad H(x)_i = (x W_g)_i + \varepsilon_i \cdot \mathrm{softplus}\big((x W_{noise})_i\big), \qquad \varepsilon_i \sim \mathcal N(0, 1).$$

Шум добавляют только на обучении. Функция потерь — квадрат ошибки общего прогноза $\sum_i G_i(x) f_i(x)$, как в формуле (1.1), плюс функция потерь балансировки $w_{importance} \cdot \mathrm{CV}(\mathrm{Importance})^2$, где $\mathrm{Importance}_i = \sum_x G_i(x)$ — суммарный вес эксперта $i$ по обучающей выборке. Обучим такую MoE на той же синтетике с $K = 4$ и $k = 2$ — без функции потерь балансировки и с $w_{importance} = 1$. В таблице «в top-$k$» — доля объектов теста, где эксперт прошел отбор, Importance — его средний вес.

In [ ]:
class NoisyTopKMoE(nn.Module):
    """Noisy top-k gating (Shazeer et al., 2017) с линейными экспертами."""
    def __init__(self, K, k):
        super().__init__()
        self.k = k
        self.experts = nn.Linear(2, K)
        self.gate = nn.Linear(2, K)          # x W_g
        self.noise = nn.Linear(2, K)         # x W_noise
        for layer in (self.gate, self.noise):
            nn.init.zeros_(layer.weight); nn.init.zeros_(layer.bias)

    def forward(self, x, noisy):
        H = self.gate(x)
        if noisy:                            # шум только на обучении
            H = H + torch.randn_like(H) * F.softplus(self.noise(x))
        top, idx = H.topk(self.k, -1)
        G = torch.full_like(H, float('-inf')).scatter(-1, idx, top).softmax(-1)   # KeepTopK, softmax
        return self.experts(x), G


def cv2(v):
    return v.var() / v.mean() ** 2           # квадрат коэффициента вариации


def train_sparse(k, w_importance, K=4, seed=0, steps=STEPS, lr=LR):
    torch.manual_seed(seed)
    model = NoisyTopKMoE(K, k).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        f, G = model(x_tr, noisy=True)
        loss = ((y_tr - (G * f).sum(-1)) ** 2).mean() + w_importance * cv2(G.sum(0))
        opt.zero_grad(); loss.backward(); opt.step()
    return model


@torch.no_grad()
def describe_sparse(model, name):
    f, G = model(x_te, noisy=False)
    top = G.argmax(-1)
    print(f'{name}: MSE смеси {((G * f).sum(-1) - y_te).pow(2).mean().item():.3f}, '
          f'CV^2(Importance) = {cv2(G.sum(0)).item():.3f}, доля нулевых весов G {(G == 0).float().mean().item():.2f}')
    print('  эксперт  в top-k  Importance  доли областей 0, 1, 2 (эксперт с наибольшим весом)')
    for e in range(G.shape[1]):
        shares = [((top == e) & (r_te == r)).sum().item() / (r_te == r).sum().item() for r in range(3)]
        print(f'  {e}     {(G[:, e] > 0).float().mean().item():.3f}    {G[:, e].mean().item():.3f}     '
              + '  '.join(f'{v:.2f}' for v in shares))


sparse0 = train_sparse(k=2, w_importance=0.0)
sparse1 = train_sparse(k=2, w_importance=1.0)
describe_sparse(sparse0, 'k = 2, без функции потерь балансировки (w_importance = 0)')
print()
describe_sparse(sparse1, 'k = 2, w_importance = 1')
print(f'\nвремя прогона тетради до этой ячейки: {time.time() - T0:.0f} с')

Половина весов $G$ нулевая по построению: из четырех экспертов отбор проходят два, и нулевой вес значит «не прошел отбор», а не «бесполезен для объекта». Сравните загрузку экспертов без функции потерь балансировки и с ней: данные и модель те же.

## Задания

1. **σ в правдоподобии.** Обучите MoE с функцией потерь (1.3) при $\sigma = 1$ и $\sigma = 2$ вместо $0.5$. Сколько экспертов получают хотя бы 1% объектов теста и как меняются MSE смеси и MSE одного эксперта? Почему с ростом $\sigma$ область 0 достается меньшему числу экспертов?
2. **Больше экспертов.** Обучите MoE с функцией потерь (1.3) и $K = 8$. Сколько экспертов из восьми получают хотя бы 1% объектов теста и чему они научились? Сравните их веса с экспертами модели с $K = 4$.
3. **Удаление по областям.** Для модели с функцией потерь (1.3) из раздела 2 посчитайте рост MSE без каждого эксперта отдельно на каждой из трех областей теста. На какой области и на сколько падает качество без самого редкого эксперта?
4. **$k = 1$ в noisy top-$k$.** Обучите `train_sparse` с $k = 1$. Что выучила gating-модель (посмотрите на `model.gate.weight`) и куда она отправляет объекты теста? Объясните это по формуле $G(x)$. Затем соберите маршрутизацию Switch Transformer (Fedus, Zoph и Shazeer, 2022): $k = 1$, а выход эксперта умножается на вероятность роутера $p_i(x) = \mathrm{softmax}(W_g x)_i$. Учится ли gating-модель теперь?
5. **Сила балансировки.** Повторите `train_sparse` с $k = 2$ и $w_{importance} = 0,\ 0.1,\ 1,\ 10$. Как меняются $\mathrm{CV}^2$, загрузка экспертов и MSE смеси?

## Выводы

- С правдоподобием смеси (1.3) эксперты специализируются: на всех пяти сидах прогноз одного эксперта с наибольшим весом точнее прогноза смеси, а веса экспертов повторяют законы своих областей. С ошибкой общего прогноза (1.1) смесь точнее, но эксперты кооперируют: по отдельности их прогнозы далеки от ответа, а веса не похожи ни на один закон, и читать такого эксперта как «закон своей области» нельзя.
- Специализация не обязана совпадать с областями, которые задал человек: параболу области 0 делят два эксперта, и gating-модель ставит границу там, где настоящей границы нет.
- Gating-модель предсказывает ответственность по одному $x$: вдали от границ $g_k(x)$ почти совпадает с $h_k(x, y)$, а у границы остается мягким, хотя $h_k(x, y)$ по ответу уже знает эксперта.
- Частота выбора — не важность: самый частый эксперт заменим соседом по параболе, а редкий незаменим на своей области. При функции потерь (1.1) MSE растет в разы без любого эксперта, даже без того, который ни разу не получает наибольший вес: эксперты исправляют ошибки друг друга.
- В top-$k$ маршрутизации нулевой вес значит «не прошел отбор». Загрузку экспертов при тех же данных меняет коэффициент функции потерь балансировки, поэтому ровная загрузка ничего не говорит о данных. При $k = 1$ в формуле Shazeer et al. gating-модель не получает градиента, и Switch Transformer обходит это множителем $p_i(x)$.